In [39]:
import pandas as pd

df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
print(df.head())

   Person ID Gender  Age            Occupation  Sleep Duration  \
0          1   Male   27     Software Engineer             6.1   
1          2   Male   28                Doctor             6.2   
2          3   Male   28                Doctor             6.2   
3          4   Male   28  Sales Representative             5.9   
4          5   Male   28  Sales Representative             5.9   

   Quality of Sleep  Physical Activity Level  Stress Level BMI Category  \
0                 6                       42             6   Overweight   
1                 6                       60             8       Normal   
2                 6                       60             8       Normal   
3                 4                       30             8        Obese   
4                 4                       30             8        Obese   

  Blood Pressure  Heart Rate  Daily Steps Sleep Disorder  
0         126/83          77         4200            NaN  
1         125/80          75      

In [40]:
import pandas as pd
from sklearn.model_selection import train_test_split


df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
df[['Systolic', 'Diastolic']] = df['Blood Pressure'].str.split('/', expand=True).astype(int)
df = df.drop(columns=['Blood Pressure', 'Person ID'])
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')

# Separating Target from Features
X = df.drop(columns=['Sleep Disorder'])
y = df['Sleep Disorder']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


Augmenting training data


In [ ]:
import numpy as np

# Recombining train temporarily so we can bootstrap it
train_df = X_train.copy()
train_df['Sleep Disorder'] = y_train

# Expanding the training data
target_size = 5500
train_df = train_df.sample(n=target_size, replace=True, random_state=42).reset_index(drop=True)

# Adding natural noise to training data
noise_profiles = {
    'Sleep Duration': 0.5, 'Quality of Sleep': 0.5, 'Physical Activity Level': 10.0, 
    'Stress Level': 0.5, 'Heart Rate': 3.0, 'Daily Steps': 800.0, 
    'Systolic': 4.0, 'Diastolic': 3.0
}
train_df['Age'] = train_df['Age'] + np.random.randint(-1, 2, size=len(train_df))
for col, std_dev in noise_profiles.items():
    train_df[col] = train_df[col] + np.random.normal(0, std_dev, size=len(train_df))


for col in ['Quality of Sleep', 'Stress Level']:
    train_df[col] = train_df[col].clip(1, 10).round(1)
for col in ['Age', 'Daily Steps', 'Systolic', 'Diastolic', 'Heart Rate', 'Physical Activity Level']:
    train_df[col] = train_df[col].clip(lower=0).round(0).astype(int)
train_df['Sleep Duration'] = train_df['Sleep Duration'].clip(lower=0).round(1)

# 4. Inject NaNs into both
def inject_nans(data):
    data.loc[data[data['Age'] > 40].sample(frac=0.15, random_state=42).index, 'Sleep Duration'] = np.nan
    data.loc[data[data['Daily Steps'] < 4000].sample(frac=0.20, random_state=42).index, 'Heart Rate'] = np.nan
    data.loc[data.sample(frac=0.03, random_state=42).index, 'BMI Category'] = np.nan
    return data

train_df = inject_nans(train_df)
X_test = inject_nans(X_test.copy())

# Separate train features and target again
X_train = train_df.drop(columns=['Sleep Disorder'])
y_train = train_df['Sleep Disorder']

print(f"Training data augmented! Train shape: {X_train.shape}, Test shape: {X_test.shape}")

Training data augmented! Train shape: (4510, 12), Test shape: (75, 12)


The Cleanup (Imputing & Encoding), then Spliting and Scaling

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Imputing
num_imputer = SimpleImputer(strategy='median')
X_train[['Sleep Duration', 'Heart Rate']] = num_imputer.fit_transform(X_train[['Sleep Duration', 'Heart Rate']])
X_test[['Sleep Duration', 'Heart Rate']] = num_imputer.transform(X_test[['Sleep Duration', 'Heart Rate']])

cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[['BMI Category']] = cat_imputer.fit_transform(X_train[['BMI Category']])
X_test[['BMI Category']] = cat_imputer.transform(X_test[['BMI Category']])

# Encoding
label_encoders = {}
for col in ['Gender', 'Occupation', 'BMI Category']:
    le = LabelEncoder()

    le.fit(X_train[col].astype(str))

    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

# Encode target separately
le_y = LabelEncoder()
y_train = le_y.fit_transform(y_train)
y_test = le_y.transform(y_test)
label_encoders['Sleep Disorder'] = le_y

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Imputation, Encoding, and Scaling complete")

Imputation, Encoding, and Scaling complete with ZERO data leakage


Model Training

In [43]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Training the model
model = RandomForestClassifier(random_state=42, n_estimators=100)
model.fit(X_train_scaled, y_train)

# Predicting and Grading
predictions = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, predictions)


print(f"Accuracy Score: {accuracy * 100:.2f}%\n")
print(classification_report(y_test, predictions, target_names=label_encoders['Sleep Disorder'].classes_))

Accuracy Score: 96.00%

              precision    recall  f1-score   support

    Insomnia       0.88      0.93      0.90        15
        None       1.00      1.00      1.00        44
 Sleep Apnea       0.93      0.88      0.90        16

    accuracy                           0.96        75
   macro avg       0.94      0.94      0.94        75
weighted avg       0.96      0.96      0.96        75



In [44]:
import joblib

# Export the trained model, the math scaler, and the text translators
joblib.dump(model, 'sleep_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoders, 'encoders.pkl')

print("Exported successfully!")

Exported successfully!


How Bias was addressed:
Class Imbalance (stratify=y) - By stratifying the split

Magnitude Bias (StandardScaler) - Scaling

Outlier Bias (.clip()) - Clipping the extreme highs and lows.